# Notebook 01 - How Machines Learn

Goes with **Lesson 03**. Run each cell with **Shift + Enter**, from top to bottom.

In this notebook you will:

1. Load the student dataset.
2. Write a rule **by hand** (traditional programming).
3. Let the computer **learn** the rule (machine learning).
4. See one small example of each type of learning.

## 1. Load the data

In [ ]:
from pathlib import Path

# The data files are in the "data" folder next to "notebooks".
# In Google Colab, upload the CSV file and it will be found in the current folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")
print("Reading data from:", DATA_DIR.resolve())

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "student_performance.csv")
print("Rows and columns:", df.shape)
df.head()

Each **row** is one student (one example). Each **column** is one piece of information.

## 2. Traditional programming: we write the rule

Let us guess a rule: *"A student passes if they study 4 hours or more."*

In [ ]:
def my_rule(hours_studied):
    if hours_studied >= 4:
        return "pass"
    else:
        return "fail"

# Use our rule on every student
df["my_rule_says"] = df["hours_studied"].apply(my_rule)

# How often is our rule correct?
correct = (df["my_rule_says"] == df["result"]).mean()
print(f"Our hand-written rule is correct for {correct:.0%} of students")

Not bad, but we **guessed** the number 4, and we only used one column. What about attendance, sleep and previous marks? Writing a perfect rule by hand is hard.

## 3. Machine learning: the computer finds the rule

First we separate the **features** (X, the questions) from the **label** (y, the answer).

In [ ]:
feature_columns = ["hours_studied", "attendance_percent", "sleep_hours", "previous_score"]

X = df[feature_columns]   # features: what we know
y = df["result"]          # label: what we want to predict

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head(3)

We keep some students **hidden** from the model so we can test it fairly later.
240 students for learning (training), 60 for testing.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training students:", len(X_train))
print("Testing students: ", len(X_test))

Now the magic line: `model.fit(X_train, y_train)` means **"learn from these examples"**.

We use an algorithm called Logistic Regression. You will learn how it works in Lesson 07. For now, just see that it learns.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)          # LEARN from 240 examples

accuracy = model.score(X_test, y_test)  # TEST on 60 students it has never seen
print(f"The learned model is correct for {accuracy:.0%} of NEW students")

## 4. Use the model for a brand-new student

In [ ]:
new_student = pd.DataFrame([{
    "hours_studied": 2,
    "attendance_percent": 75,
    "sleep_hours": 6,
    "previous_score": 55,
}])

prediction = model.predict(new_student)[0]
print("Prediction for the new student:", prediction)

**Try it:** change `hours_studied` from 2 to 8 in the cell above and run it again.

## 5. A quick look at the four types of learning

### 5a. Supervised learning - regression (predict a NUMBER)

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()
reg.fit(X_train, df.loc[X_train.index, "final_score"])

predicted_marks = reg.predict(new_student)[0]
print(f"Predicted final marks for the new student: {predicted_marks:.1f}")

### 5b. Unsupervised learning - clustering (no labels)

We give the computer **only** hours studied and sleep hours, with **no** pass/fail answers, and ask it to find 3 groups.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

habits = df[["hours_studied", "sleep_hours"]]
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["group"] = kmeans.fit_predict(habits)

print(df["group"].value_counts().sort_index())

plt.scatter(df["hours_studied"], df["sleep_hours"], c=df["group"])
plt.xlabel("Hours studied")
plt.ylabel("Sleep hours")
plt.title("3 groups found WITHOUT any labels")
plt.show()

The computer found groups by itself. A person then decides what each group means (for example "low study", "medium study", "high study").

### 5c. Semi-supervised learning (only a few labels)

We pretend that we only know the result for **10 percent** of training students. The rest are marked `-1`, which means "unknown".

In [ ]:
import numpy as np
from sklearn.semi_supervised import SelfTrainingClassifier

y_train_numbers = (y_train == "pass").astype(int).to_numpy()   # pass = 1, fail = 0

rng = np.random.RandomState(42)
y_few_labels = y_train_numbers.copy()
hide = rng.rand(len(y_few_labels)) > 0.10        # hide about 90 percent
y_few_labels[hide] = -1

print("Students with a known label:", (y_few_labels != -1).sum(), "out of", len(y_few_labels))

semi = SelfTrainingClassifier(LogisticRegression(max_iter=1000))
semi.fit(X_train, y_few_labels)

y_test_numbers = (y_test == "pass").astype(int)
print(f"Accuracy with only a few labels: {semi.score(X_test, y_test_numbers):.0%}")

Even with very few labels, the model labels the rest of the data itself and still does well.

### 5d. Reinforcement learning (trial, error and reward)

There is no dataset here. An **agent** must choose one of 3 snack machines. Each machine gives a reward (a snack) with a hidden chance. The agent does not know the chances. It learns by **trying** and keeping score.

- 90 percent of the time it picks the machine that looks best so far (**exploit**).
- 10 percent of the time it tries a random machine (**explore**).

In [ ]:
import random
random.seed(1)

hidden_chance = [0.2, 0.5, 0.8]     # the agent does NOT know these
total_reward = [0, 0, 0]
times_tried = [0, 0, 0]

for step in range(1000):
    if random.random() < 0.1 or 0 in times_tried:
        choice = random.randrange(3)                        # explore
    else:
        averages = [total_reward[i] / times_tried[i] for i in range(3)]
        choice = averages.index(max(averages))              # exploit

    reward = 1 if random.random() < hidden_chance[choice] else 0
    total_reward[choice] += reward
    times_tried[choice] += 1

for i in range(3):
    print(f"Machine {i}: tried {times_tried[i]:4} times, average reward {total_reward[i] / times_tried[i]:.2f}")

The agent learned that **machine 2** is the best and used it most - without anyone telling it. This is the idea behind robots learning to walk and AlphaGo learning to play.

## Summary

| You did | Type |
|---|---|
| Wrote `my_rule` by hand | Traditional programming |
| `fit` on features + labels, then `predict` | Supervised learning (classification and regression) |
| Found groups with no labels | Unsupervised learning |
| Learned with only 10 percent labels | Semi-supervised learning |
| Learned by trying and getting rewards | Reinforcement learning |

Go back to **Lesson 03** and do the **Check** section.